# 08 — SVGD-inferens: Bayesiansk parameter-estimering

I denne notebook bruger SVGD (Stein Variational Gradient Descent) til at beregne den fulde posterior fordeling for two-island modellens parametre.

**Spørgsmål jeg gerne vil undersøger:**

- Hvad er SVGD-posterioren for to-parameter two-island modellen?
- Hvornår konvergerer SVGD, og hvad hjælper?
- Hvad er forskellen mellem SVGD med MoM-prior vs. generisk prior?
- Hvad sker der med posterioren for IM-modellen med tre parametre?

In [ ]:
# ALTID importer phasic først
from phasic import (
    Graph, with_ipv, GaussPrior, HalfCauchyPrior,
    Adam, ExpStepSize, ExpRegularization, clear_caches,
    StateIndexer, Property
)
import numpy as np
import jax.numpy as jnp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ecdf

%config InlineBackend.figure_format = 'svg'
np.random.seed(42)
sns.set_palette('tab10')
plt.rcParams['figure.dpi'] = 120
clear_caches()

## SVGD basics: enkelt-parameter koalescent

jeg starter med svgd-basics eksempel og demonstrerer effekten af prior, learning rate og Adam-optimizeren.

In [ ]:
# Koalescent med et parameter 
nr_samples = 4

@with_ipv([nr_samples]+[0]*(nr_samples-1))
def coalescent_1param(state):
    transitions = []
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            new = state.copy()
            new[i] -= 1; new[j] -= 1; new[i+j+1] += 1
            transitions.append([new, [state[i]*(state[j]-same)/(1+same)]])
    return transitions

graph = Graph(coalescent_1param)

true_theta = [7]
graph.update_weights(true_theta)
observed_data = graph.sample(10000)

# Histogram vs. teoretisk PDF
fig, ax = plt.subplots(figsize=(7, 4))
t_plot = np.linspace(0, max(observed_data)*1.1, 200)
ax.hist(observed_data, bins=60, density=True, alpha=0.5, label='Data')
ax.plot(t_plot, graph.pdf(t_plot), lw=2, label=f'Sand PDF (θ={true_theta[0]})')
ax.set_xlabel('TMRCA'); ax.set_ylabel('Tæthed')
ax.set_title(f'Simulerede data (n={len(observed_data):,})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- SVGD med default indstillinger ---
svgd_default = graph.svgd(observed_data, n_iterations=200)
svgd_default.summary()
svgd_default.plot_convergence()

In [ ]:
# --- MoM-prior + ExpStepSize schedule (vejlederens anbefalede workflow) ---
mom = graph.method_of_moments(observed_data)
print(f"MoM startpunkt: theta={mom.theta}, std={mom.std}")

step_schedule = ExpStepSize(first_step=0.05, last_step=0.005, tau=30.0)

svgd_mom = graph.svgd(
    observed_data,
    prior=mom.prior,
    learning_rate=step_schedule,
    n_iterations=150
)
svgd_mom.summary()
svgd_mom.plot_convergence()

In [ ]:
# --- Adam optimizer: automatisk adaptiv learning rate ---
svgd_adam = graph.svgd(
    observed_data,
    prior=mom.prior,
    optimizer=Adam(learning_rate=0.25),
    n_iterations=100
)
svgd_adam.summary()
svgd_adam.plot_ci(true_theta=true_theta)

In [ ]:
# Sammenlign: default vs. MoM-prior vs. Adam
methods = [
    ('Default', svgd_default),
    ('MoM + ExpStepSize', svgd_mom),
    ('Adam', svgd_adam),
]

fig, axes = plt.subplots(1, len(methods), figsize=(13, 4), sharey=True)
for ax, (name, svgd_obj) in zip(axes, methods):
    res = svgd_obj.get_results()
    # Tegn particle-distribution
    particles = np.array(res.get('theta_particles', [res['theta_mean']]))
    if particles.ndim == 1:
        particles = particles.reshape(-1, 1)
    from scipy.stats import norm as sp_norm
    mu = float(res['theta_mean'])
    sd = float(res['theta_std'])
    x_r = np.linspace(mu - 4*sd, mu + 4*sd, 200)
    ax.plot(x_r, sp_norm.pdf(x_r, mu, sd), lw=2)
    ax.axvline(true_theta[0], color='red', ls='--', lw=1.5, label='Sand')
    ax.axvline(mu, color='blue', ls=':', lw=1.5, label=f'MAP={mu:.2f}')
    ax.set_title(name); ax.set_xlabel('θ')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Tæthed')
plt.suptitle('Posterior for θ: tre SVGD-konfigurationer', y=1.02)
plt.tight_layout()
plt.show()

## Posterior predictive check

Jeg verificerer at den estimerede model faktisk passer til data ved at sammenligne ECDF og model-CDF.

In [ ]:
# Posterior predictive check (fra vejlederens svgd-basics)
res_adam = svgd_adam.get_results()
graph.update_weights([float(res_adam['theta_mean'])])

emp_cdf = ecdf(observed_data)
x_vals = emp_cdf.cdf.quantiles
emp_probs = emp_cdf.cdf.probabilities

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(observed_data, bins=60, density=True, alpha=0.5, label='Data')
ax1.plot(x_vals, graph.pdf(x_vals), lw=2, color='C1', label='Estimeret PDF')
graph.update_weights(true_theta)
ax1.plot(x_vals, graph.pdf(x_vals), '--', lw=1.5, color='C2', label='Sand PDF')
ax1.set_xlabel('TMRCA'); ax1.set_ylabel('Tæthed'); ax1.set_title('PDF')
ax1.legend(fontsize=9)

graph.update_weights([float(res_adam['theta_mean'])])
model_cdf = graph.cdf(x_vals)
ax2.plot(x_vals, emp_probs, lw=2, label='ECDF (data)')
ax2.plot(x_vals, model_cdf, '--', lw=2, label='Model CDF')
ax3 = ax2.twinx()
ax3.plot(x_vals, model_cdf - emp_probs, lw=1, color='C2', alpha=0.6, label='Forskel')
ax3.set_ylabel('CDF - ECDF', color='C2')
ax3.tick_params(axis='y', colors='C2')
ax3.grid(False)
ax2.set_xlabel('TMRCA'); ax2.set_ylabel('F(t)'); ax2.set_title('CDF + forskel')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
graph.update_weights(true_theta)  # reset

## Two-island modellen: two-parameter SVGD

Jeg bruger two-island StateIndexer-grafen fra notebook 07 til fuld Bayesiansk inferens med SVGD.

In [ ]:
# Genbyg two-island graf
nr_samples_ti = 2
indexer = StateIndexer(
    descendants=[
        Property('pop1', min_value=0, max_value=nr_samples_ti),
        Property('pop2', min_value=0, max_value=nr_samples_ti),
        Property('in_pop', min_value=1, max_value=2),
    ]
)
initial = [0] * indexer.state_length
initial[indexer.descendants.props_to_index(pop1=1, pop2=0, in_pop=1)] = nr_samples_ti

@with_ipv(initial)
def coalescent_islands(state):
    transitions = []
    if state[indexer.descendants.indices()].sum() <= 1:
        return transitions
    for i in range(indexer.descendants.state_length):
        if state[i] == 0: continue
        props_i = indexer.descendants.index_to_props(i)
        for j in range(i, indexer.descendants.state_length):
            if state[j] == 0: continue
            props_j = indexer.descendants.index_to_props(j)
            if props_j.in_pop != props_i.in_pop: continue
            same = int(i == j)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            child = state.copy()
            child[i] -= 1; child[j] -= 1
            dp1 = props_i.pop1 + props_j.pop1
            dp2 = props_i.pop2 + props_j.pop2
            if dp1 <= nr_samples_ti and dp2 <= nr_samples_ti:
                k = indexer.descendants.props_to_index(pop1=dp1, pop2=dp2, in_pop=props_i.in_pop)
                child[k] += 1
                transitions.append([child, [state[i]*(state[j]-same)/(1+same), 0]])
        if state[i] > 0:
            child = state.copy()
            other_pop = 2 if props_i.in_pop == 1 else 1
            child[i] -= 1
            k = indexer.descendants.props_to_index(pop1=props_i.pop1, pop2=props_i.pop2, in_pop=other_pop)
            child[k] += 1
            transitions.append([child, [0, state[i]]])
    return transitions

graph_ti = Graph(coalescent_islands)

# Simuler data med sande parametre
true_theta_ti = [0.7, 0.9]
graph_ti.update_weights(true_theta_ti)
data_ti = graph_ti.sample(2000)
print(f"Simulerede {len(data_ti)} observationer")
print(f"Sand theta: {true_theta_ti}")

In [ ]:
# MoM startpunkt
mom_ti = graph_ti.method_of_moments(data_ti)
print(f"MoM estimat: {mom_ti.theta}")
print(f"MoM std:     {mom_ti.std}")

# SVGD med MoM-prior og Adam (vejlederens anbefalede kombination)
svgd_ti = graph_ti.svgd(
    data_ti,
    prior=mom_ti.prior,
    optimizer=Adam(learning_rate=0.25),
    n_iterations=100
)
svgd_ti.summary()

In [ ]:
# Konvergensplot og pairwise posterior
svgd_ti.plot_convergence()
svgd_ti.plot_pairwise(true_theta=true_theta_ti)

In [ ]:
svgd_ti.plot_trace()

## Hvad sker der med posterioren for varierende $M$?

**Eksperiment:** Estimer $M$ for tre sande værdier og sammenlign
posterior-bredden. Hvornår er posterioren smal vs. bred?

In [ ]:
M_true_vals = [0.3, 1.0, 3.0]
N_true_fixed = 0.7  # 1/N fast
n_obs_exp = 1500

fig, axes = plt.subplots(1, len(M_true_vals), figsize=(13, 4))
from scipy.stats import norm as sp_norm

for ax, M_true in zip(axes, M_true_vals):
    true_t = [N_true_fixed, M_true/2]  # [1/N, M/2]
    graph_ti.update_weights(true_t)
    data_m = graph_ti.sample(n_obs_exp)

    mom_m = graph_ti.method_of_moments(data_m)

    svgd_m = graph_ti.svgd(
        data_m,
        prior=mom_m.prior,
        optimizer=Adam(learning_rate=0.25),
        n_iterations=80
    )
    res = svgd_m.get_results()
    mu1 = float(res['theta_mean'][1])  # M/2-estimat
    sd1 = float(res['theta_std'][1])

    x_r = np.linspace(max(0, mu1 - 4*sd1), mu1 + 4*sd1, 200)
    ax.plot(x_r, sp_norm.pdf(x_r, mu1, sd1), lw=2, label=f'Posterior M/2')
    ax.axvline(M_true/2, color='red', ls='--', lw=2, label=f'Sand M/2={M_true/2}')
    ax.axvline(mu1, color='blue', ls=':', lw=1.5, label=f'MAP={mu1:.2f}')
    ax.set_title(f'M={M_true} (M/2={M_true/2})')
    ax.set_xlabel('M/2')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Tæthed')
plt.suptitle(f'Posterior for M/2: varierende sand M (1/N={N_true_fixed}, n={n_obs_exp})', y=1.02)
plt.tight_layout()
plt.show()
print("Observation:")
print("- Lav M → bred posterior (sværere at estimere, færre koalescentshændelser på tværs).")
print("- Høj M → smal posterior (signal er klarere i data).")

## Prior-sensitivitet: hvad sker der med forkert prior?

**Eksperiment:** Kør SVGD med en misspecificeret prior (for smal, for bred, og centreret forkert) og se om Adam kan korrigere.

In [ ]:
graph.update_weights(true_theta)
data_prior_test = graph.sample(5000)

prior_scenarios = [
    ('MoM-prior (korrekt)', graph.method_of_moments(data_prior_test).prior),
    ('Forkert center CI=[1,3]', GaussPrior(ci=[1, 3])),
    ('Meget bred CI=[0.1, 20]', GaussPrior(ci=[0.1, 20])),
    ('For smal CI=[6.5, 7.5]', GaussPrior(ci=[6.5, 7.5])),
]

fig, axes = plt.subplots(1, len(prior_scenarios), figsize=(14, 4), sharey=True)

for ax, (name, prior) in zip(axes, prior_scenarios):
    svgd_p = graph.svgd(
        data_prior_test,
        prior=prior,
        optimizer=Adam(learning_rate=0.25),
        n_iterations=150
    )
    res = svgd_p.get_results()
    mu = float(res['theta_mean'])
    sd = float(res['theta_std'])

    x_r = np.linspace(max(0.1, mu - 5*sd), mu + 5*sd, 200)
    ax.plot(x_r, sp_norm.pdf(x_r, mu, sd), lw=2)
    ax.axvline(true_theta[0], color='red', ls='--', lw=2, label='Sand')
    ax.axvline(mu, color='blue', ls=':', lw=1.5, label=f'MAP={mu:.2f}')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('θ')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Tæthed')
plt.suptitle('Prior-sensitivitet med Adam optimizer', y=1.02)
plt.tight_layout()
plt.show()
print("Observation:")
print("- Adam er robust over for misspecificerede priors med tilstrækkelig data.")
print("- En meget smal prior kan begrænse posterioren selv med meget data.")
print("- MoM-prior giver det bedste startpunkt og smaleste posterior CI.")

## SVGD med regularisering: hvornår hjælper det?

Regularisering trækker partikler mod de empiriske momenter i de tidlige iterationer og kan stabilisere konvergens for komplekse modeller.

In [ ]:
# Sammenlign: med og uden regularisering for two-island
graph_ti.update_weights(true_theta_ti)
data_reg = graph_ti.sample(1000)
mom_reg = graph_ti.method_of_moments(data_reg)

reg_schedule = ExpRegularization(first_reg=10.0, last_reg=0.1, tau=20.0)

svgd_no_reg = graph_ti.svgd(
    data_reg, prior=mom_reg.prior,
    optimizer=Adam(learning_rate=0.25), n_iterations=100
)
svgd_reg = graph_ti.svgd(
    data_reg, prior=mom_reg.prior,
    optimizer=Adam(learning_rate=0.25),
    regularization=reg_schedule,
    nr_moments=2,
    n_iterations=100
)

print("Uden regularisering:")
svgd_no_reg.summary()
print("\nMed regularisering:")
svgd_reg.summary()

In [ ]:
# Konvergensplots side om side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (svgd_obj, label) in zip(axes,
    [(svgd_no_reg, 'Uden regularisering'), (svgd_reg, 'Med regularisering')]):
    res = svgd_obj.get_results()
    # Kald plot_convergence og hent axes
    svgd_obj.plot_convergence()
    plt.title(label)
    plt.show()

print("Observation:")
print("- Regularisering kan hjælpe med at holde partikler samlet i de tidlige iterationer.")
print("- For velkonvergerede modeller er effekten typisk lille.")

## 7. MoM vs. SVGD: direkte sammenligning

**Eksperiment:** Kør 30 repetitioner med varierende stikprøvestørrelser og sammenlign RMSE for MoM og SVGD MAP-estimat.

In [ ]:
n_obs_compare = [100, 300, 1000, 3000]
n_reps_compare = 20  # Reduceret for beregningstid
true_t_comp = [0.7, 0.9]

compare_results = []

for n_obs_c in n_obs_compare:
    mom_rmse0, mom_rmse1 = [], []
    svgd_rmse0, svgd_rmse1 = [], []

    for rep in range(n_reps_compare):
        graph_ti.update_weights(true_t_comp)
        d = graph_ti.sample(n_obs_c)

        # MoM
        try:
            m = graph_ti.method_of_moments(d)
            if m.success:
                mom_rmse0.append((float(m.theta[0]) - true_t_comp[0])**2)
                mom_rmse1.append((float(m.theta[1]) - true_t_comp[1])**2)
        except Exception: pass

        # SVGD MAP
        try:
            mom_sv = graph_ti.method_of_moments(d)
            sv = graph_ti.svgd(d, prior=mom_sv.prior,
                               optimizer=Adam(0.25), n_iterations=60)
            res = sv.get_results()
            svgd_rmse0.append((float(res['theta_map'][0]) - true_t_comp[0])**2)
            svgd_rmse1.append((float(res['theta_map'][1]) - true_t_comp[1])**2)
        except Exception: pass

    compare_results.append({
        'n_obs': n_obs_c,
        'MoM_RMSE_theta0': np.sqrt(np.mean(mom_rmse0)) if mom_rmse0 else np.nan,
        'MoM_RMSE_theta1': np.sqrt(np.mean(mom_rmse1)) if mom_rmse1 else np.nan,
        'SVGD_RMSE_theta0': np.sqrt(np.mean(svgd_rmse0)) if svgd_rmse0 else np.nan,
        'SVGD_RMSE_theta1': np.sqrt(np.mean(svgd_rmse1)) if svgd_rmse1 else np.nan,
    })

df_compare = pd.DataFrame(compare_results)
print("MoM vs. SVGD RMSE:")
print(df_compare.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col0, col1, title in zip(
    axes,
    ['MoM_RMSE_theta0', 'MoM_RMSE_theta1'],
    ['SVGD_RMSE_theta0', 'SVGD_RMSE_theta1'],
    ['RMSE for θ₀ (1/N)', 'RMSE for θ₁ (M/2)']
):
    ax.loglog(df_compare['n_obs'], df_compare[col0], 'o-', lw=2, label='MoM')
    ax.loglog(df_compare['n_obs'], df_compare[col1], 's--', lw=2, label='SVGD MAP')
    ax.set_xlabel('n_obs'); ax.set_ylabel('RMSE')
    ax.set_title(title); ax.legend()

plt.tight_layout()
plt.show()
print("Observation:")
print("- For store n_obs konvergerer MoM og SVGD mod samme RMSE.")
print("- For lille n_obs kan SVGD have fordel pga. fuld likelihood.")
print("- Største forskel forventes for θ₁ (M), da migrationsrate er sværere at estimere.")

## Opsummering

| Eksperiment | Nøgleresultat |
|---|---|
| SVGD basics | MoM-prior + Adam giver bedst konvergens |
| Posterior predictive | ECDF og model-CDF stemmer godt overens |
| Posterior vs. M | Lav M → bred posterior; høj M → smal |
| Prior-sensitivitet | Adam robust over for misspecificeret prior |
| MoM vs. SVGD | Konvergerer for store n; SVGD bedre for lille n |

**Næste notebook:** 09_baboon_analysis.ipynb — inferens på baviandata.